# Meyer–Wallach Entanglement on IQM Spark

**Authors:** Koło Naukowe Axion

**Goal:** measure **Meyer–Wallach (MW) entanglement** on the IQM Spark (Odra) quantum computer without statevectors, using **local X/Y/Z Pauli tomography** per qubit.

**Protocol**
1. For each ansatz × depth × random parameter sample, bind random $\theta \in [0, 2\pi)$.
2. Run **3 circuits** per sample (Z, X, Y basis rotations + `measure_all`).
3. From counts, estimate $\langle X\rangle$, $\langle Y\rangle$, $\langle Z\rangle$ per qubit; compute per-qubit purity and MW score.
4. Aggregate mean/std/sem/min/max over samples.

**Scale:** 2 ansatze × 3 depths × 20 samples × 3 bases = **360 circuits** (4096 shots each) by default.


## 1. Imports & Configuration

In [ ]:
from qbanknote.paths import ensure_importable
ensure_importable()

import csv
import getpass
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

try:
    from iqm.qiskit_iqm import transpile_to_IQM as _iqm_transpile
    from iqm.qiskit_iqm.iqm_backend import IQMBackendBase as _IQMBackendBase
except ImportError:
    _iqm_transpile = None
    _IQMBackendBase = None

NUM_QUBITS = 5
DEPTHS = [2, 4, 6]
ANSATZES = ("ansatz_odra", "ansatz_simulator")
SEED = 42
N_SAMPLES = 20
SHOTS = 4096
OPTIMIZATION_LEVEL = 1
MAX_CIRCUITS_PER_JOB = 275
IQM_URL = os.environ.get("IQM_URL", "https://odra5.e-science.pl/").strip()
NOTEBOOK_DIR = Path(".").resolve()

n_jobs = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * 3
print(
    f"Planned run: {len(ANSATZES)} ansatze x {len(DEPTHS)} depths x "
    f"{N_SAMPLES} samples x 3 bases = {n_jobs} circuits ({SHOTS} shots each)"
)


## 2. Ansatz Definitions

- **Ansatz 1 — Ring (simulator-optimized).** RY · CRX · RX · CRY ring with a q0-incident reverse trim on the last macro-layer.
- **Ansatz 2 — Odra (IQM-Spark adapted).** RY · RZ/CZ · RX · RY/CZ ring with the same trim.


In [ ]:
from qbanknote.ansatzes import (
    ansatz_trimmed_reverse_q0_param_count,
    odra_ansatz as ansatz_odra,
    simulator_ansatz as ansatz_simulator,
)

for depth in DEPTHS:
    n_params = ansatz_trimmed_reverse_q0_param_count(NUM_QUBITS, depth)
    print(f"depth={depth}: n_params={n_params}")


## 3. Meyer–Wallach & Measurement Helpers

In [ ]:
from qbanknote.iqm import connect_to_iqm_backend, transpile_for_backend, normalize_counts, run_circuits_on_backend
from qbanknote.metrics import (
    BASIS_ORDER,
    BasisName,
    add_basis_measurement,
    bitstring_qubit_value,
    qubit_expectation_from_counts,
    mw_score_from_bloch,
    estimate_mw_from_hardware_counts,
    compute_iqm_mw_scores,
    meyer_wallach_score,
    single_qubit_reduced_density,
    run_mw_self_check as run_self_check,
)

DEFAULT_IQM_URL = "https://odra5.e-science.pl/"
DEFAULT_SHOTS = 4096
DEFAULT_N_SAMPLES = 20
DEFAULT_SEED = 42
DEFAULT_OPTIMIZATION_LEVEL = 1
DEFAULT_MAX_CIRCUITS_PER_JOB = 275


## 4. Self-check (no hardware)

In [ ]:
run_self_check()

## 5. Connect to IQM Spark

Set `IQM_TOKEN` in your environment or enter the token when prompted.


In [ ]:
iqm_backend = connect_to_iqm_backend(IQM_URL)
print(f"Connected to backend: {iqm_backend}  (n_qubits = {iqm_backend.num_qubits})")


## 6. Hardware sweep

In [ ]:
stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = NOTEBOOK_DIR / f"iqm_mw_{stamp}"
output_dir.mkdir(parents=True, exist_ok=True)

ansatz_fns = {
    "ansatz_odra": ansatz_odra,
    "ansatz_simulator": ansatz_simulator,
}

summary_rows = []
score_rows = []

for depth in DEPTHS:
    for ansatz_name in ANSATZES:
        print(f"Running {ansatz_name} depth={depth} ...")
        depth_seed = SEED + depth * 1000 + (1 if ansatz_name == "ansatz_simulator" else 0)
        result = compute_iqm_mw_scores(
            iqm_backend,
            ansatz_fns[ansatz_name],
            n_qubits=NUM_QUBITS,
            depth=depth,
            n_samples=N_SAMPLES,
            seed=depth_seed,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=None,
            max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
        )
        summary_rows.append(
            {
                "ansatz": ansatz_name,
                "depth": depth,
                "n_qubits": result["n_qubits"],
                "n_params": result["n_params"],
                "n_samples": result["n_samples"],
                "shots": SHOTS,
                "seed": depth_seed,
                "mw_avg": result["mw_avg"],
                "mw_std": result["mw_std"],
                "mw_sem": result["mw_sem"],
                "mw_min": result["mw_min"],
                "mw_max": result["mw_max"],
            }
        )
        for bloch in result["bloch_rows"]:
            row = {
                "ansatz": ansatz_name,
                "depth": depth,
                "sample_index": int(bloch["sample_index"]),
                "mw_score": bloch["mw_score"],
            }
            for key, value in bloch.items():
                if key not in ("sample_index", "mw_score"):
                    row[key] = value
            score_rows.append(row)

summary_path = output_dir / "iqm_mw_results.csv"
scores_path = output_dir / "iqm_mw_scores.csv"
manifest_path = output_dir / "run_manifest.json"

with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

score_fieldnames = list(score_rows[0].keys()) if score_rows else []
with scores_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=score_fieldnames)
    writer.writeheader()
    writer.writerows(score_rows)

manifest = {
    "created_utc": datetime.now(tz=timezone.utc).isoformat(),
    "backend": str(iqm_backend),
    "iqm_url": IQM_URL,
    "source_notebook": "evaluation_and_comparison/iqm_meyer_wallach.ipynb",
    "method": "local_xyz_tomography",
    "n_qubits": NUM_QUBITS,
    "depths": list(DEPTHS),
    "ansatzes": list(ANSATZES),
    "n_samples": N_SAMPLES,
    "shots": SHOTS,
    "seed": SEED,
    "optimization_level": OPTIMIZATION_LEVEL,
    "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    "total_circuits": n_jobs,
    "outputs": [summary_path.name, scores_path.name],
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

results_df = pd.DataFrame(summary_rows)
scores_df = pd.DataFrame(score_rows)
print(f"\nSaved outputs to {output_dir}")
results_df


## 7. Results & Plots

In [ ]:
print("MW comparison (higher = more entanglement):")
print("depth | ansatz           | mw_avg   | mw_std")
print("-" * 50)
for _, row in results_df.iterrows():
    print(
        f"{int(row['depth']):>5} | {row['ansatz']:<16} | "
        f"{row['mw_avg']:.6f} | {row['mw_std']:.6f}"
    )

fig, ax = plt.subplots(figsize=(8, 4))
for ansatz_name in ANSATZES:
    sub = results_df[results_df["ansatz"] == ansatz_name].sort_values("depth")
    ax.plot(sub["depth"], sub["mw_avg"], marker="o", label=ansatz_name)
    ax.fill_between(
        sub["depth"],
        sub["mw_avg"] - sub["mw_sem"],
        sub["mw_avg"] + sub["mw_sem"],
        alpha=0.2,
    )
ax.set_xlabel("Depth")
ax.set_ylabel("Mean Meyer–Wallach score")
ax.set_title("MW entanglement vs depth on IQM Spark")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
